# YOLOv8 Smart Labeling Tool 

## Project Overview
This notebook contains the complete source code for the **Smart Labeler**, a Human-in-the-Loop annotation tool designed to accelerate dataset creation for object detection. 

### Core Logic Explained
Below are the specific logic snippets responsible for the AI's intelligence. These are integrated into the main application class in the code block below.

#### 1. Model Loading Strategy
The system automatically searches for the best available model. It prioritizes the latest custom-trained version (`train_vX`) to ensure the annotation assistant gets smarter over time. If no custom model exists, it falls back to the pre-trained `yolov8n.pt` (Transfer Learning).

```python
def _load_last_model(self):
    # Scans 'runs' folder for the highest version number (e.g., train_v5)
    folders = [f for f in os.listdir(runs_path) if f.startswith("train")]
    latest = sorted(folders, key=get_version)[-1]
    
    if exists(latest):
        self.model = YOLO(latest_weights) # Load Custom
    else:
        self.model = YOLO("yolov8n.pt") # Load COCO Pre-trained
```

#### 2. Inference (AI Assistance)
When a new image is loaded, the model predicts bounding boxes. These are displayed as "Auto" boxes (dashed lines) for the user to verify.

```python
def predict_current(self):
    results = self.model(self.current_image_path)
    for box in results.boxes:
        # Extract coordinates and class
        x1, y1, x2, y2 = box.xyxy[0]
        cls = int(box.cls[0])
        # Add to canvas as 'Auto' (is_auto=True)
        self.bboxes.append([x1, y1, x2, y2, cls, conf, True])
```

#### 3. Training Engine
The training loop runs on a separate thread to keep the GUI responsive. It uses a **Virtual Split** strategy, creating temporary `.txt` files for training and validation sets to ensure scientific rigor (80/20 split) without duplicating image files.

```python
def train_model(self, epochs, batch, patience, split_pct):
    # 1. Generate random split (.txt files)
    train_files, val_files = create_virtual_split(split_pct)
    
    # 2. Update data.yaml configuration
    update_yaml(train_files, val_files)
    
    # 3. Start YOLO training
    model.train(
        data='data.yaml',
        epochs=epochs,
        patience=patience, # Early Stopping
        project='dataset_labeled/runs'
    )
```

### Credits
The Graphical User Interface (GUI), including the Tkinter layout and event handling, was developed with the assistance of **Gemini** to ensure a robust and user-friendly annotation workflow.

In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from PIL import Image, ImageTk
import os
import glob
import random
import shutil
import threading
from ultralytics import YOLO
import yaml

# --- CONFIGURATION ---
RAW_DATA_PATH = "dataset_raw" 
TRAIN_DATA_PATH = "dataset_labeled"

CLASS_NAMES = [
    "door_open", 
    "trunk_open", 
    "hood_open"
]
CLASS_COLORS = ["red", "blue", "orange"]

class AutoLabelerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("YOLO Smart Labeler")
        self.root.geometry("1600x950")
        
        # --- STATE VARIABLES ---
        self.current_image_path = None
        self.current_image = None 
        self.tk_image = None      
        self.scale = 1.0
        
        self.bboxes = []          
        self.current_class = 0
        self.is_drawing = False
        self.start_x = 0
        self.start_y = 0
        self.model = None         
        self.is_training = False
        
        # Statistics
        self.stat_images_processed = 0  
        self.stat_auto_generated = 0    
        self.stat_auto_deleted = 0      
        self.stat_manual_added = 0      
        
        # Review Mode
        self.review_mode = False
        self.review_list = []
        
        self._setup_directories()
        
        # 1. Load Data
        self.raw_image_list = self._get_raw_images()
        print(f"Found {len(self.raw_image_list)} raw images.")
        
        # 2. Setup UI
        self._setup_ui()
        
        # 3. Load Model
        self._load_last_model()
        
        # Start
        self.load_next_image()

    def _setup_directories(self):
        # We store everything in 'train' physically.
        # The split happens logically using .txt files during training.
        os.makedirs(f"{TRAIN_DATA_PATH}/images/train", exist_ok=True)
        os.makedirs(f"{TRAIN_DATA_PATH}/labels/train", exist_ok=True)
        
        # Initial dummy data.yaml (will be overwritten during training)
        data_yaml = {
            'path': os.path.abspath(TRAIN_DATA_PATH),
            'train': 'images/train',
            'val': 'images/train', 
            'names': {i: name for i, name in enumerate(CLASS_NAMES)}
        }
        with open(f"{TRAIN_DATA_PATH}/data.yaml", 'w') as f:
            yaml.dump(data_yaml, f)

    def _setup_ui(self):
        # --- LEFT: CANVAS ---
        self.canvas_frame = tk.Frame(self.root, bg="gray")
        self.canvas_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        self.canvas = tk.Canvas(self.canvas_frame, bg="black", cursor="cross")
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        self.canvas.bind("<ButtonPress-1>", self.on_mouse_down)
        self.canvas.bind("<B1-Motion>", self.on_mouse_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_mouse_up)
        self.canvas.bind("<Button-3>", self.on_right_click) 

        # --- RIGHT: CONTROLS ---
        self.controls = tk.Frame(self.root, width=350, bg="white")
        self.controls.pack(side=tk.RIGHT, fill=tk.Y)
        
        # 1. Top Counters
        self.lbl_remaining = tk.Label(self.controls, text=f"RAW IMAGES LEFT: {len(self.raw_image_list)}", 
                                      font=("Arial", 14, "bold"), fg="red", bg="#eeeeee")
        self.lbl_remaining.pack(fill=tk.X, pady=5, ipady=5)

        # 2. Box List
        self.box_list_frame = tk.Frame(self.controls, bg="white")
        self.box_list_frame.pack(fill=tk.X, pady=5)
        self.btn_toggle_list = tk.Button(self.box_list_frame, text="[+] Show Box List", command=self.toggle_box_list, bg="#dddddd")
        self.btn_toggle_list.pack(fill=tk.X)
        self.tree_frame = tk.Frame(self.box_list_frame) 
        cols = ("Class", "Conf", "Source")
        self.tree = ttk.Treeview(self.tree_frame, columns=cols, show='headings', height=6)
        self.tree.heading("Class", text="Class")
        self.tree.heading("Conf", text="Conf")
        self.tree.heading("Source", text="Source")
        self.tree.column("Class", width=80); self.tree.column("Conf", width=50); self.tree.column("Source", width=60)
        self.tree.pack(fill=tk.X, padx=5)

        # 3. Tools
        tk.Label(self.controls, text="Select Tool:", font=("Arial", 10, "bold")).pack(pady=5)
        self.class_var = tk.IntVar(value=0)
        for i, name in enumerate(CLASS_NAMES):
            tk.Radiobutton(self.controls, text=f"{name} ({CLASS_COLORS[i]})", 
                           variable=self.class_var, value=i, command=self.change_class).pack(anchor="w", padx=20)

        # 4. Actions
        tk.Frame(self.controls, height=2, bg="black").pack(fill=tk.X, pady=10)
        self.btn_save = tk.Button(self.controls, text="SAVE & NEXT (Space)", bg="#ccffcc", command=self.save_and_next)
        self.btn_save.pack(fill=tk.X, padx=10, pady=2)
        self.btn_discard = tk.Button(self.controls, text="DISCARD (Del)", bg="#ffcccc", command=self.discard_image)
        self.btn_discard.pack(fill=tk.X, padx=10, pady=2)
        self.btn_undo = tk.Button(self.controls, text="UNDO Last (Ctrl+Z)", command=self.undo_last_action)
        self.btn_undo.pack(fill=tk.X, padx=10, pady=2)
        self.btn_reselect = tk.Button(self.controls, text="♻️ RESELECT / REVIEW", bg="#ffffcc", command=self.open_reselect_menu)
        self.btn_reselect.pack(fill=tk.X, padx=10, pady=10)

        # 5. Training Settings
        self.train_settings_frame = tk.LabelFrame(self.controls, text="Training Settings", bg="white", font=("Arial", 10, "bold"))
        self.train_settings_frame.pack(fill=tk.X, padx=10, pady=10)
        
        # Grid layout for inputs
        # Row 0: Epochs
        tk.Label(self.train_settings_frame, text="Epochs:", bg="white").grid(row=0, column=0, sticky="e", padx=5)
        self.entry_epochs = tk.Entry(self.train_settings_frame, width=5)
        self.entry_epochs.insert(0, "50")
        self.entry_epochs.grid(row=0, column=1, padx=5)

        # Row 1: Batch
        tk.Label(self.train_settings_frame, text="Batch Size:", bg="white").grid(row=1, column=0, sticky="e", padx=5)
        self.entry_batch = tk.Entry(self.train_settings_frame, width=5)
        self.entry_batch.insert(0, "16")
        self.entry_batch.grid(row=1, column=1, padx=5)

        # Row 2: Patience
        tk.Label(self.train_settings_frame, text="Patience:", bg="white").grid(row=2, column=0, sticky="e", padx=5)
        self.entry_patience = tk.Entry(self.train_settings_frame, width=5)
        self.entry_patience.insert(0, "15") 
        self.entry_patience.grid(row=2, column=1, padx=5)

        # Row 3: Split %
        tk.Label(self.train_settings_frame, text="Val Split %:", bg="white").grid(row=3, column=0, sticky="e", padx=5)
        self.entry_split = tk.Entry(self.train_settings_frame, width=5)
        self.entry_split.insert(0, "20") # Default 20%
        self.entry_split.grid(row=3, column=1, padx=5)

        # Train Button
        self.btn_train = tk.Button(self.train_settings_frame, text="START TRAINING", bg="lightblue", command=self.start_training_thread)
        self.btn_train.grid(row=4, column=0, columnspan=2, pady=10, sticky="ew", padx=5)

        # 6. AI & Stats
        tk.Label(self.controls, text="AI Assistance", font=("Arial", 11, "bold")).pack(pady=2)
        self.auto_predict_var = tk.BooleanVar(value=True)
        tk.Checkbutton(self.controls, text="Enable Auto-Predict", variable=self.auto_predict_var).pack()
        self.btn_predict = tk.Button(self.controls, text="Predict Current (Manual)", command=self.predict_current)
        self.btn_predict.pack(fill=tk.X, padx=10, pady=2)
        
        self.status_label = tk.Label(self.controls, text="Status: Ready", fg="blue")
        self.status_label.pack(pady=5)

        # 7. Session Stats
        self.stats_frame = tk.Frame(self.controls, bg="#e6f3ff", bd=2, relief=tk.GROOVE)
        self.stats_frame.pack(fill=tk.X, padx=10, pady=10)
        tk.Label(self.stats_frame, text="SESSION PERFORMANCE", font=("Arial", 10, "bold"), bg="#e6f3ff").pack(pady=5)
        
        self.lbl_stat_imgs = tk.Label(self.stats_frame, text="Images Viewed: 0", bg="#e6f3ff", anchor="w")
        self.lbl_stat_imgs.pack(fill=tk.X, padx=10)
        self.lbl_stat_acc = tk.Label(self.stats_frame, text="AI Accuracy: 100%", font=("Arial", 12, "bold"), fg="green", bg="#e6f3ff")
        self.lbl_stat_acc.pack(fill=tk.X, padx=10, pady=5)
        tk.Frame(self.stats_frame, height=1, bg="gray").pack(fill=tk.X, padx=5, pady=5)
        self.lbl_stat_good = tk.Label(self.stats_frame, text="✅ Auto-Boxes Kept: 0", fg="green", bg="#e6f3ff", anchor="w")
        self.lbl_stat_good.pack(fill=tk.X, padx=10)
        self.lbl_stat_del = tk.Label(self.stats_frame, text="❌ Auto-Boxes Deleted: 0", fg="red", bg="#e6f3ff", anchor="w")
        self.lbl_stat_del.pack(fill=tk.X, padx=10)
        self.lbl_stat_add = tk.Label(self.stats_frame, text="✏️ Manual Boxes Added: 0", fg="orange", bg="#e6f3ff", anchor="w")
        self.lbl_stat_add.pack(fill=tk.X, padx=10, pady=(0, 10))

        # 8. General Stats
        self.stats_text = tk.Text(self.controls, height=4, width=30)
        self.stats_text.pack(padx=10, pady=5)
        self.update_stats()

        self.root.bind("<space>", lambda e: self.save_and_next())
        self.root.bind("<Delete>", lambda e: self.discard_image())
        self.root.bind("<Control-z>", lambda e: self.undo_last_action())

    # --- UI HELPERS ---
    def toggle_box_list(self):
        if self.tree_frame.winfo_ismapped():
            self.tree_frame.pack_forget()
            self.btn_toggle_list.config(text="[+] Show Box List")
        else:
            self.tree_frame.pack(fill=tk.X)
            self.btn_toggle_list.config(text="[-] Hide Box List")

    def update_box_list_ui(self):
        for item in self.tree.get_children():
            self.tree.delete(item)
        for box in self.bboxes:
            cls_name = CLASS_NAMES[box[4]]
            conf_str = f"{box[5]:.2f}" if box[5] is not None else "-"
            src_str = "Auto" if box[6] else "Manual"
            self.tree.insert('', tk.END, values=(cls_name, conf_str, src_str))
        
        auto_kept = max(0, self.stat_auto_generated - self.stat_auto_deleted)
        total_final_boxes = auto_kept + self.stat_manual_added
        
        if total_final_boxes > 0:
            accuracy = (auto_kept / total_final_boxes) * 100.0
        else:
            if self.stat_auto_deleted > 0: accuracy = 0.0
            else: accuracy = 100.0

        acc_color = "green" if accuracy > 85 else "orange" if accuracy > 60 else "red"

        self.lbl_stat_imgs.config(text=f"Images Viewed: {self.stat_images_processed}")
        self.lbl_stat_acc.config(text=f"AI Accuracy: {accuracy:.1f}%", fg=acc_color)
        self.lbl_stat_good.config(text=f"✅ Auto-Boxes Kept: {auto_kept}")
        self.lbl_stat_del.config(text=f"❌ Auto-Boxes Deleted: {self.stat_auto_deleted}")
        self.lbl_stat_add.config(text=f"✏️ Manual Boxes Added: {self.stat_manual_added}")

    # --- MODEL LOGIC ---
    def _load_last_model(self):
        runs_path = os.path.join(TRAIN_DATA_PATH, "runs")
        best_model = None
        if os.path.exists(runs_path):
            folders = [f for f in os.listdir(runs_path) if f.startswith("train")]
            def get_v(name):
                try: return int(name.split('_v')[-1])
                except: return 0
            folders.sort(key=get_v, reverse=True)
            if folders:
                latest_folder = folders[0]
                model_path = os.path.join(runs_path, latest_folder, "weights", "best.pt")
                if os.path.exists(model_path):
                    best_model = model_path
                    print(f"Loaded latest model: {latest_folder}")

        if best_model:
            try:
                self.model = YOLO(best_model)
                self.status_label.config(text="Status: Custom Model Loaded", fg="green")
                self.btn_predict.config(state=tk.NORMAL)
            except:
                self.model = YOLO("yolov8n.pt")
        else:
            self.model = YOLO("yolov8n.pt") 
            self.btn_predict.config(state=tk.DISABLED)

    def start_training_thread(self):
        if self.is_training: return
        self.status_label.config(text="Status: TRAINING... (Stats Paused)", fg="red")
        self.is_training = True
        
        try:
            epochs = int(self.entry_epochs.get())
            batch = int(self.entry_batch.get())
            patience = int(self.entry_patience.get())
            split_pct = float(self.entry_split.get())
            if split_pct < 0 or split_pct > 100: raise ValueError
        except ValueError:
            messagebox.showerror("Error", "Please enter valid numbers.\nSplit must be 0-100.")
            self.is_training = False
            return

        t = threading.Thread(target=self.train_model, args=(epochs, batch, patience, split_pct))
        t.daemon = True
        t.start()

    def train_model(self, epochs, batch, patience, split_pct):
        print(f"--- STARTING TRAINING (E={epochs}, B={batch}, Pat={patience}, Split={split_pct}%) ---")
        try:
            abs_data_path = os.path.abspath(TRAIN_DATA_PATH)
            runs_dir = os.path.join(abs_data_path, "runs")
            os.makedirs(runs_dir, exist_ok=True)
            
            # 1. Determine Run Name
            existing_runs = [d for d in os.listdir(runs_dir) if d.startswith("train_v")]
            next_id = 1
            if existing_runs:
                ids = []
                for r in existing_runs:
                    try: ids.append(int(r.split("_v")[-1]))
                    except: pass
                if ids: next_id = max(ids) + 1
            
            run_name = f"train_v{next_id}"
            
            # 2. GENERATE VIRTUAL SPLIT (Create .txt files)
            # Find all images currently in images/train
            img_dir = os.path.join(abs_data_path, "images", "train")
            all_images = glob.glob(os.path.join(img_dir, "*.jpg")) + \
                         glob.glob(os.path.join(img_dir, "*.png")) + \
                         glob.glob(os.path.join(img_dir, "*.jpeg"))
            
            random.shuffle(all_images)
            
            split_ratio = split_pct / 100.0
            split_index = int(len(all_images) * (1 - split_ratio))
            
            train_files = all_images[:split_index]
            val_files = all_images[split_index:]
            
            # Edge case: If 0% split or no validation files, use train for val to avoid crash
            if not val_files:
                val_files = train_files

            train_txt_path = os.path.join(abs_data_path, "autosplit_train.txt")
            val_txt_path = os.path.join(abs_data_path, "autosplit_val.txt")
            
            with open(train_txt_path, 'w') as f:
                f.write('\n'.join(train_files))
                
            with open(val_txt_path, 'w') as f:
                f.write('\n'.join(val_files))

            # 3. Update YAML to point to .txt files instead of folders
            data_yaml = {
                'path': abs_data_path,
                'train': train_txt_path, # Points to list of files
                'val': val_txt_path,     # Points to list of files
                'names': {i: name for i, name in enumerate(CLASS_NAMES)}
            }
            yaml_path = os.path.join(abs_data_path, "data.yaml")
            with open(yaml_path, 'w') as f:
                yaml.dump(data_yaml, f)

            # 4. START TRAINING
            new_model = YOLO("yolov8n.pt") 
            results = new_model.train(
                data=yaml_path,
                epochs=epochs,
                imgsz=640,
                batch=batch,
                patience=patience,
                project=runs_dir,
                name=run_name,
                exist_ok=True 
            )
            
            new_weights = os.path.join(runs_dir, run_name, "weights", "best.pt")
            
            # 5. GENERATE VALIDATION VISUALS
            # If we had validation images, predict on them and save to run folder
            if split_pct > 0 and val_files:
                print("Generating validation visualisations...")
                val_vis_dir = os.path.join(runs_dir, run_name, "validation_predictions")
                best_model = YOLO(new_weights)
                # Run prediction on the validation list
                best_model.predict(
                    source=val_txt_path, 
                    project=runs_dir,
                    name=f"{run_name}/validation_predictions", # Creates subdirectory
                    save=True,
                    conf=0.5,
                    exist_ok=True
                )
            
            # Metrics
            map50_95 = results.box.map
            precision = results.box.mp
            recall = results.box.mr
            
            self.model = YOLO(new_weights)
            
            self.root.after(0, lambda: self._on_train_success(run_name, map50_95, precision, recall))
            
        except Exception as e:
            error_msg = str(e)
            print(f"Training failed: {error_msg}")
            self.root.after(0, lambda: self.status_label.config(text=f"Error: {error_msg}", fg="red"))
        self.is_training = False

    def _on_train_success(self, run_name, map_val, prec_val, rec_val):
        self.status_label.config(text=f"Status: Done ({run_name})", fg="green")
        self.btn_predict.config(state=tk.NORMAL)
        msg = f"Training Completed: {run_name}\n\nMetrics:\nmAP: {map_val:.3f}\nPrec: {prec_val:.3f}\nRec: {rec_val:.3f}\n\nVisuals saved in: runs/{run_name}/validation_predictions"
        messagebox.showinfo("Training Results", msg)

    # --- INTERACTION ---
    def on_mouse_down(self, event):
        self.is_drawing = True
        self.start_x = event.x
        self.start_y = event.y

    def on_mouse_drag(self, event):
        if self.is_drawing:
            self.canvas.delete("temp_box")
            self.canvas.create_rectangle(self.start_x, self.start_y, event.x, event.y, 
                                         outline=CLASS_COLORS[self.current_class], width=2, tags="temp_box")

    def on_mouse_up(self, event):
        self.is_drawing = False
        self.canvas.delete("temp_box")
        if abs(self.start_x - event.x) > 5 and abs(self.start_y - event.y) > 5:
            self.bboxes.append([self.start_x, self.start_y, event.x, event.y, 
                                self.current_class, None, False])
            self.stat_manual_added += 1 
            self.redraw_boxes()
            self.update_box_list_ui()

    def on_right_click(self, event):
        x, y = event.x, event.y
        deleted_index = -1
        for i in range(len(self.bboxes) - 1, -1, -1):
            bx1, by1, bx2, by2, _, _, _ = self.bboxes[i]
            left, right = min(bx1, bx2), max(bx1, bx2)
            top, bottom = min(by1, by2), max(by1, by2)
            if left <= x <= right and top <= y <= bottom:
                deleted_index = i
                break
        
        if deleted_index != -1:
            box = self.bboxes.pop(deleted_index)
            is_auto = box[6]
            if is_auto:
                self.stat_auto_deleted += 1
            else:
                self.stat_manual_added = max(0, self.stat_manual_added - 1)
            self.redraw_boxes()
            self.update_box_list_ui()
        else:
            self.undo_last_action()

    def undo_last_action(self):
        if self.bboxes:
            box = self.bboxes.pop()
            if not box[6]: 
                self.stat_manual_added = max(0, self.stat_manual_added - 1)
            self.redraw_boxes()
            self.update_box_list_ui()

    def redraw_boxes(self):
        self.canvas.delete("box")
        for box in self.bboxes:
            x1, y1, x2, y2, cls, conf, is_auto = box
            color = CLASS_COLORS[cls]
            dash = (4, 4) if is_auto else None
            width = 2 if not is_auto else 1
            self.canvas.create_rectangle(x1, y1, x2, y2, outline=color, width=width, dash=dash, tags="box")
            label_txt = CLASS_NAMES[cls]
            if conf: label_txt += f" {conf:.2f}"
            self.canvas.create_text(x1, y1-10, text=label_txt, fill=color, tags="box")

    # --- FILE OPS ---
    def load_next_image(self):
        if self.review_mode:
            count = len(self.review_list)
            self.lbl_remaining.config(text=f"REVIEW LEFT: {count}", fg="blue")
            if not self.review_list:
                messagebox.showinfo("Review Done", "Returning to Normal Mode.")
                self.review_mode = False
                self.load_next_image()
                return
            self.current_image_path = self.review_list.pop(0)
        else:
            count = len(self.raw_image_list)
            self.lbl_remaining.config(text=f"RAW IMAGES LEFT: {count}", fg="red")
            if not self.raw_image_list:
                messagebox.showinfo("Done", "No more raw images found!")
                return
            self.current_image_path = self.raw_image_list.pop(0)

        self.bboxes = []
        img = Image.open(self.current_image_path)
        self.original_size = img.size 
        
        cw = self.canvas.winfo_width() if self.canvas.winfo_width() > 1 else 1000
        ch = self.canvas.winfo_height() if self.canvas.winfo_height() > 1 else 700
        
        w_ratio = cw / img.width
        h_ratio = ch / img.height
        self.scale = min(w_ratio, h_ratio) * 0.9 
        
        new_size = (int(img.width * self.scale), int(img.height * self.scale))
        self.current_image = img.resize(new_size, Image.Resampling.LANCZOS)
        self.tk_image = ImageTk.PhotoImage(self.current_image)
        
        self.canvas.delete("all")
        self.canvas.create_image(cw//2, ch//2, image=self.tk_image, anchor=tk.CENTER)
        
        self.offset_x = (cw//2) - (new_size[0]//2)
        self.offset_y = (ch//2) - (new_size[1]//2)

        if self.review_mode:
            self.load_existing_labels()
        elif self.model and self.auto_predict_var.get():
            self.predict_current()
            
        self.update_box_list_ui()

    def predict_current(self):
        if not self.model: return
        results = self.model(self.current_image_path)
        new_auto_count = 0
        self.bboxes = [] 
        for r in results:
            boxes = r.boxes
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                if cls < len(CLASS_NAMES):
                    cx1 = (x1 * self.scale) + self.offset_x
                    cy1 = (y1 * self.scale) + self.offset_y
                    cx2 = (x2 * self.scale) + self.offset_x
                    cy2 = (y2 * self.scale) + self.offset_y
                    self.bboxes.append([cx1, cy1, cx2, cy2, cls, conf, True])
                    new_auto_count += 1
        self.stat_auto_generated += new_auto_count
        self.redraw_boxes()
        self.update_box_list_ui()

    def load_existing_labels(self):
        filename = os.path.basename(self.current_image_path)
        label_filename = os.path.splitext(filename)[0] + ".txt"
        label_path = os.path.join(TRAIN_DATA_PATH, "labels", "train", label_filename)
        
        if os.path.exists(label_path):
            w_orig, h_orig = self.original_size
            with open(label_path, 'r') as f:
                for line in f:
                    try:
                        parts = list(map(float, line.split()))
                        cls = int(parts[0])
                        cx, cy, w, h = parts[1], parts[2], parts[3], parts[4]
                        
                        x_center = cx * w_orig
                        y_center = cy * h_orig
                        width_px = w * w_orig
                        height_px = h * h_orig
                        
                        x1 = ((x_center - width_px/2) * self.scale) + self.offset_x
                        y1 = ((y_center - height_px/2) * self.scale) + self.offset_y
                        x2 = ((x_center + width_px/2) * self.scale) + self.offset_x
                        y2 = ((y_center + height_px/2) * self.scale) + self.offset_y
                        
                        self.bboxes.append([x1, y1, x2, y2, cls, None, False])
                    except: pass
        self.redraw_boxes()

    def save_and_next(self):
        self.stat_images_processed += 1 
        filename = os.path.basename(self.current_image_path)
        
        # Always move to 'images/train' physically
        dest_img_path = os.path.join(TRAIN_DATA_PATH, "images/train", filename)
        if not self.review_mode:
            shutil.move(self.current_image_path, dest_img_path)
        
        label_filename = os.path.splitext(filename)[0] + ".txt"
        dest_lbl_path = os.path.join(TRAIN_DATA_PATH, "labels", "train", label_filename)
        
        with open(dest_lbl_path, "w") as f:
            for box in self.bboxes:
                x1 = (box[0] - self.offset_x) / self.scale
                y1 = (box[1] - self.offset_y) / self.scale
                x2 = (box[2] - self.offset_x) / self.scale
                y2 = (box[3] - self.offset_y) / self.scale
                cls = box[4]
                
                w_orig, h_orig = self.original_size
                x1, x2 = max(0, x1), min(w_orig, x2)
                y1, y2 = max(0, y1), min(h_orig, y2)
                
                b_center_x = (x1 + x2) / 2.0 / w_orig
                b_center_y = (y1 + y2) / 2.0 / h_orig
                b_width = abs(x2 - x1) / w_orig
                b_height = abs(y2 - y1) / h_orig
                f.write(f"{cls} {b_center_x} {b_center_y} {b_width} {b_height}\n")
        
        self.update_stats()
        self.load_next_image()

    def discard_image(self):
        for box in self.bboxes:
            is_auto = box[6]
            if is_auto:
                self.stat_auto_generated = max(0, self.stat_auto_generated - 1)
            else:
                self.stat_manual_added = max(0, self.stat_manual_added - 1)

        try:
            if self.review_mode:
                os.remove(self.current_image_path)
                # Remove label
                base_name = os.path.splitext(os.path.basename(self.current_image_path))[0] + ".txt"
                parent = os.path.dirname(os.path.dirname(self.current_image_path)) 
                lbl = os.path.join(parent, "labels", "train", base_name)
                if os.path.exists(lbl): os.remove(lbl)
            else:
                os.remove(self.current_image_path)
        except: pass
        
        self.update_stats()
        self.load_next_image()

    def change_class(self):
        self.current_class = self.class_var.get()

    def open_reselect_menu(self):
        top = tk.Toplevel(self.root)
        top.title("Reselect")
        for i, name in enumerate(CLASS_NAMES):
            tk.Button(top, text=f"Review: {name}", command=lambda c=i: self.start_review_mode(c, top)).pack(fill=tk.X)
    
    def start_review_mode(self, class_id, window):
        window.destroy()
        self.status_label.config(text="Scanning...", fg="orange")
        self.root.update()
        
        label_dir = os.path.join(TRAIN_DATA_PATH, "labels", "train")
        image_dir = os.path.join(TRAIN_DATA_PATH, "images", "train")
        found = []
        for tf in glob.glob(os.path.join(label_dir, "*.txt")):
            with open(tf) as f:
                if any(int(l.split()[0]) == class_id for l in f):
                    bn = os.path.splitext(os.path.basename(tf))[0]
                    for ext in ['.jpg','.png']: 
                        img_p = os.path.join(image_dir, bn+ext)
                        if os.path.exists(img_p): 
                            found.append(img_p); break
        if found:
            self.review_mode = True
            self.review_list = found
            self.lbl_remaining.config(text=f"REVIEW LEFT: {len(found)}", fg="blue")
            self.load_next_image()
        else: messagebox.showinfo("None found", "No images with that label.")

    def _get_raw_images(self):
        exts = ['*.jpg', '*.jpeg', '*.png']
        files = []
        for ext in exts: files.extend(glob.glob(os.path.join(RAW_DATA_PATH, '**', ext), recursive=True))
        random.shuffle(files)
        return files

    def update_stats(self):
        label_dir = os.path.join(TRAIN_DATA_PATH, "labels", "train")
        if not os.path.exists(label_dir): return
        
        txt_files = glob.glob(os.path.join(label_dir, "*.txt"))
        counts = {name: 0 for name in CLASS_NAMES}
        for tf in txt_files:
            with open(tf, 'r') as f:
                for line in f:
                    try:
                        c_id = int(line.split()[0])
                        if c_id < len(CLASS_NAMES): counts[CLASS_NAMES[c_id]] += 1
                    except: pass
                        
        stats_str = f"Total Labeled: {len(txt_files)}\n"
        for name, count in counts.items(): stats_str += f"{name}: {count}\n"
        self.stats_text.delete(1.0, tk.END)
        self.stats_text.insert(tk.END, stats_str)

if __name__ == "__main__":
    if not os.path.exists(RAW_DATA_PATH): os.makedirs(RAW_DATA_PATH, exist_ok=True)
    root = tk.Tk()
    app = AutoLabelerApp(root)
    root.mainloop()

Found 471 raw images.
Loaded latest model: train_v6

image 1/1 /home/robin/Data science code/Data_Project/dataset_raw/camera_top_view/20260121165608_1769018044128_frame001200.jpg: 384x640 4 door_opens, 1 trunk_open, 1 hood_open, 56.8ms
Speed: 3.1ms preprocess, 56.8ms inference, 22.7ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /home/robin/Data science code/Data_Project/dataset_raw/camera_top_view/20260121165001_1769018013728_frame000960.jpg: 384x640 (no detections), 70.3ms
Speed: 4.0ms preprocess, 70.3ms inference, 8.1ms postprocess per image at shape (1, 3, 384, 640)
